# Feature Engineering: Housing Dataset Milestone 3

This notebook focuses on creating new features form the cleaned Ames Housing dataset. The goal is to transform existing variables into features that will provide useful information for predicting 'SalePrice'. Many of the variables are categorical, so those need to be made into values a model is able to use. After all features have been finalized, scalization will be considered because some of the variables are ranking, some will be one-hot encoded, and some are dollar values that are quite large. 

There are some features that could be redundant based on EDA, so I will also be exploring those further to ensure their usefulness for predictive modeling. 

In [1]:
import pandas as pd
import numpy as np

housing = pd.read_csv('ames_housing_cleaned.csv')      

## 1. Total Bath Equivalent

The dataset contains separate features for full and half bathrooms. I will combine these features into one, 'Total.Bath.Equiv'. This will be calculated by 'Full.Bath + (.5 * Half.Bath)'.

In [2]:
housing['Total.Bath.Equiv'] = (
    housing['Full.Bath'] + (0.5 * housing ['Half.Bath'])
)

housing[['Full.Bath', 'Half.Bath', 'Total.Bath.Equiv']].head()

,Full.Bath,Half.Bath,Total.Bath.Equiv
0,1,0,1.0
1,1,0,1.0
2,1,1,1.5
3,2,1,2.5
4,2,1,2.5


In [3]:
housing = housing.drop(columns=['Full.Bath', 'Half.Bath'])

## 2. Create Has.Pool Feature

There is a feature that indicates the size of a properties pool area. Since only 13 out of 2917 properties have pools, but the pools seem to have an effect on 'SalePrice', I will be creating a binary feature that indicates whether a property has or does not have a pool. This will be indicated by a 1 for has a pool and 0 for does not have a pool.

I will then remove the Pool.Area from the cleaned dataset. 

In [4]:
housing['Has.Pool'] = (housing['Pool.Area'] > 0).astype(int)

housing['Has.Pool'].value_counts()

Has.Pool
0    2917
1      13
Name: count, dtype: int64

In [5]:
housing = housing.drop(columns=['Pool.Area'])

'Pool.Area' in housing.columns

False

## 3. Create House Age

Using 'Year.Built' and 'Yr.Sold', I will create a variable that describes the age of the house. Once I create this variable, I will check to see if it has any effect on the house price to see if it will be useful for modeling. Then I will check for redundancy among the features that were used to create it to see if I will be keeping those features.        

In [6]:
housing['House.Age'] = housing['Yr.Sold'] - housing['Year.Built']

In [7]:
housing[['Year.Built', 'Yr.Sold', 'House.Age']].head()

,Year.Built,Yr.Sold,House.Age
0,1960,2010,50
1,1961,2010,49
2,1958,2010,52
3,1968,2010,42
4,1997,2010,13


In [8]:
housing['House.Age'].describe()

count    2930.000000
mean       36.434130
std        30.291357
min        -1.000000
25%         7.000000
50%        34.000000
75%        54.000000
max       136.000000
Name: House.Age, dtype: float64

In [9]:
housing[['House.Age', 'SalePrice']].corr(method="spearman")

,House.Age,SalePrice
House.Age,1.000000,-0.678953
SalePrice,-0.678953,1.000000


In [10]:
housing[["Year.Built", "House.Age"]].corr(method="spearman")

,Year.Built,House.Age
Year.Built,1.000000,-0.996249
House.Age,-0.996249,1.000000


### Decision

Based on this correlation, 'Year.Built' will be removed because it shares such a strong inverse relationship with 'House.Age'.

In [11]:
housing = housing.drop(columns=["Year.Built"])

"Year.Built" in housing.columns

False

## 4. Create Total Finished Basement Area

There is currently a 'Total.Bsmt.SF' and 'Bsmt.Unf.SF'. The first feature indicates the total square footage of the basement and second indicates the square footage of the basement that is unfinished. The 'Finished.Bsmt.SF' will be calculated by 'Total.Bsmt.SF' - 'Bsmt.Unf.Sf'. Then I will determine whether I need to keep all three features for the model or if there is too much redundancy.

In [12]:
housing["Finished.Bsmt.SF"] = (
    housing["Total.Bsmt.SF"] - housing["Bsmt.Unf.SF"]
)

housing[['Total.Bsmt.SF', 'Bsmt.Unf.SF', 'Finished.Bsmt.SF']].head()

,Total.Bsmt.SF,Bsmt.Unf.SF,Finished.Bsmt.SF
0,1080.0,441.0,639.0
1,882.0,270.0,612.0
2,1329.0,406.0,923.0
3,2110.0,1045.0,1065.0
4,928.0,137.0,791.0


In [13]:
housing[["Finished.Bsmt.SF", "SalePrice"]].corr(method="spearman")

,Finished.Bsmt.SF,SalePrice
Finished.Bsmt.SF,1.000000,0.331102
SalePrice,0.331102,1.000000


In [14]:
housing[
    ["Total.Bsmt.SF", "Bsmt.Unf.SF", "Finished.Bsmt.SF", "SalePrice"]
].corr(method="spearman")

,Total.Bsmt.SF,Bsmt.Unf.SF,Finished.Bsmt.SF,SalePrice
Total.Bsmt.SF,1.000000,0.331012,0.458779,0.605888
Bsmt.Unf.SF,0.331012,1.000000,-0.598368,0.163236
Finished.Bsmt.SF,0.458779,-0.598368,1.000000,0.331102
SalePrice,0.605888,0.163236,0.331102,1.000000


### Decision

Based on the correlations, I am going to keep the new feature along with the 'Total.Bsmt.SF' and remove the 'Bsmt.Unf.SF' feature. 

In [15]:
housing = housing.drop(columns=["Bsmt.Unf.SF"])

In [16]:
"Bsmt.Unf.SF" in housing.columns

False

## 5. Evaluate Living Area Features

The dataset currently contains three features that indicate living area square footage. There is 'Gr.Liv.Area' which indicates a total livable square footage. There is 'X1st.Liv.Area' and 'X2nd.Liv.Area' which indicate the livable square footage for the 1st and 2nd floors. I want to determine if all three of these features are necessary for the model. 

In [17]:
housing[
    ['X1st.Flr.SF', 'X2nd.Flr.SF', 'Gr.Liv.Area', 'SalePrice']
].corr(method='spearman')

,X1st.Flr.SF,X2nd.Flr.SF,Gr.Liv.Area,SalePrice
X1st.Flr.SF,1.000000,-0.326250,0.491441,0.581536
X2nd.Flr.SF,-0.326250,1.000000,0.604345,0.246861
Gr.Liv.Area,0.491441,0.604345,1.000000,0.723342
SalePrice,0.581536,0.246861,0.723342,1.000000


### Decision

Based on the correlations, I will keep 'Gr.Liv.Area' and remove the other square footage indicators. This reduces redundancy in the living area features. 

In [18]:
housing = housing.drop(columns=['X1st.Flr.SF', 'X2nd.Flr.SF'])

In [19]:
['X1st.Flr.SF' in housing.columns, 'X2nd.Flr.SF' in housing.columns, 'Gr.Liv.Area' in housing.columns]



[False, False, True]

## 6. Evaluate Remodeling Age

The dataset currently indicates the year the house was remodeled. To make this feature more meaningful for modeling, I am going to create a 'Years.Since.Remodel' feature and see if that has a greater effect on the 'SalePrice'.

In [20]:
housing['Years.Since.Remodel'] = (
    housing['Yr.Sold'] - housing['Year.Remod.Add']
)

In [21]:
housing[
    ['House.Age', 'Years.Since.Remodel', 'Year.Remod.Add', 'SalePrice']
].corr(method='spearman')

,House.Age,Years.Since.Remodel,Year.Remod.Add,SalePrice
House.Age,1.000000,0.714799,-0.701375,-0.678953
Years.Since.Remodel,0.714799,1.000000,-0.991202,-0.605573
Year.Remod.Add,-0.701375,-0.991202,1.000000,0.601454
SalePrice,-0.678953,-0.605573,0.601454,1.000000


### Decision

Based on the correlations, I will remove the original feature 'Year.Remod.Add' and keep the new feature 'Years.Since.Remodal'.

In [22]:
housing = housing.drop(columns=['Year.Remod.Add'])

'Year.Remodel.Add' in housing.columns

False

## 7. Create Total Outdoor Area

The dataset contains two separate measures for wood deck area and porch area. I am going to combine them to create a single measure of outdoor area, 'Total.Outdoor.SF'.

In [23]:
housing['Total.Outdoor.SF'] = (
    housing['Wood.Deck.SF'] + housing['Open.Porch.SF']
)

In [24]:
housing = housing.drop(
    columns=['Wood.Deck.SF', 'Open.Porch.SF']
)

## 8. Encode Central Air

'Central.Air' is a binary categorical variable that indicates whether a property has central air. It currently has Y/N values. It needs to be encoded as 1 for central air or 0 for no central air.

In [25]:
housing['Central.Air'] = housing['Central.Air'].map({
    'Y': 1,
    'N': 0
})

In [26]:
housing['Central.Air'].value_counts()

Central.Air
1    2734
0     196
Name: count, dtype: int64

## 9. Ordinal Encode Functional

The dataset currently contains a feature 'Functional' which indicates the degree of functionality of the home. Since this feature has a natural order, I will convert the orginal order into numerical features using ordinal encoding. 

In [27]:
functional_map = {
    'Typ': 0,
    'Min1': 1,
    'Min2': 2,
    'Mod': 3,
    'Maj1': 4,
    'Maj2': 5,
    'Sev': 6,
    'Sal': 7
}

housing['Functional'] = housing['Functional'].map(functional_map)

In [28]:
housing['Functional'].value_counts().sort_index()

Functional
0    2728
1      65
2      70
3      35
4      19
5       9
6       2
7       2
Name: count, dtype: int64

## 10. Ordinal Encode Garage.Finish

'Garage.Finish' currently indicates the level of finish inside the garage. The categories are NONE, UNF, RFN, FIN. Since there is an order to these categories, I will convert them to numerical values using ordinal encoding.

In [29]:
garage_finish = pd.read_csv('ames_housing_cleaned.csv')['Garage.Finish']

housing['Garage.Finish'] = garage_finish.map({
    'None': 0,
    'Unf': 1,
    'RFn': 2,
    'Fin': 3
})

In [30]:
housing['Garage.Finish'].value_counts(dropna=False).sort_index()

Garage.Finish
1.0    1231
2.0     812
3.0     728
NaN     159
Name: count, dtype: int64

In [31]:
housing['Garage.Finish'] = housing['Garage.Finish'].fillna(0).astype(int)

In [32]:
housing['Garage.Finish'].value_counts().sort_index()

Garage.Finish
0     159
1    1231
2     812
3     728
Name: count, dtype: int64

### Explanation

During this encoding, I had an issue changing the 'None' value to 0. After the initial encoding and checking the original values in for each category, I changed all the remaining values to 0.

## 10. Evaluate One-Hot Encoding Needs

There are currently 15 additional categorical features that I want to keep for modeling. Before encoding those features, I want to determine if they will add too many features to the final modeling dataset. Once I explore that, I will make a decision about keeping them all, or dropping some.

In [33]:
categorical_features = [
    'MS.SubClass',
    'MS.Zoning',
    'Lot.Shape',
    'Land.Contour',
    'Lot.Config',
    'Neighborhood',
    'Condition.1',
    'Roof.Style',
    'Exterior.1st',
    'Exterior.2nd',
    'Exter.Qual',
    'Foundation',
    'Electrical',
    'Garage.Type',
    'Sale.Condition'
]

housing[categorical_features].nunique()

MS.SubClass       16
MS.Zoning          7
Lot.Shape          4
Land.Contour       4
Lot.Config         5
Neighborhood      28
Condition.1        9
Roof.Style         6
Exterior.1st      16
Exterior.2nd      17
Exter.Qual         4
Foundation         6
Electrical         5
Garage.Type        6
Sale.Condition     6
dtype: int64

### Investigate Exterior.1st and Exterior.2nd for Redundancy

In [34]:
housing[['Exterior.1st', 'Exterior.2nd']].value_counts().head(20)

Exterior.1st  Exterior.2nd
VinylSd       VinylSd         1006
MetalSd       MetalSd          437
HdBoard       HdBoard          383
Wd Sdng       Wd Sdng          358
Plywood       Plywood          186
CemntBd       CmentBd          124
BrkFace       BrkFace           44
WdShing       Wd Shng           41
HdBoard       Plywood           41
AsbShng       AsbShng           35
Stucco        Stucco            33
BrkFace       Wd Sdng           21
Wd Sdng       Plywood           21
Plywood       Brk Cmn           18
Wd Sdng       Wd Shng           18
WdShing       Plywood           10
VinylSd       Wd Shng            9
BrkFace       Plywood            8
Wd Sdng       HdBoard            7
Plywood       HdBoard            6
Name: count, dtype: int64

### Decision

Since there are so many instances where the materials occur simultaneously for the 1st and 2nd feature, I don't think it is necessary to use both features for modeling. 

In [35]:
housing = housing.drop(columns=['Exterior.2nd'])

## 11. One-Hot Encode Features

There are 14 nominal features that need to be converted to numerical values. They do not have a natural numerical order, so one-hot encoding is the best option for this process. The first category will be dropped to avoid perfect multicollinearity.

In [36]:
housing = pd.get_dummies(
    housing,
    columns=['Neighborhood'],
    drop_first=True,
    dtype=int
)

In [37]:
[c for c in housing.columns if c.startswith('Neighborhood_')]

['Neighborhood_Blueste',
 'Neighborhood_BrDale',
 'Neighborhood_BrkSide',
 'Neighborhood_ClearCr',
 'Neighborhood_CollgCr',
 'Neighborhood_Crawfor',
 'Neighborhood_Edwards',
 'Neighborhood_Gilbert',
 'Neighborhood_Greens',
 'Neighborhood_GrnHill',
 'Neighborhood_IDOTRR',
 'Neighborhood_Landmrk',
 'Neighborhood_MeadowV',
 'Neighborhood_Mitchel',
 'Neighborhood_NAmes',
 'Neighborhood_NPkVill',
 'Neighborhood_NWAmes',
 'Neighborhood_NoRidge',
 'Neighborhood_NridgHt',
 'Neighborhood_OldTown',
 'Neighborhood_SWISU',
 'Neighborhood_Sawyer',
 'Neighborhood_SawyerW',
 'Neighborhood_Somerst',
 'Neighborhood_StoneBr',
 'Neighborhood_Timber',
 'Neighborhood_Veenker']

In [38]:
housing = pd.get_dummies(
    housing,
    columns=['Roof.Style'],
    drop_first=True,
    dtype=int
)
    

In [39]:
[c for c in housing.columns if c.startswith('Roof.Style_')]

['Roof.Style_Gable',
 'Roof.Style_Gambrel',
 'Roof.Style_Hip',
 'Roof.Style_Mansard',
 'Roof.Style_Shed']

In [40]:
housing = pd.get_dummies(
    housing,
    columns=['MS.SubClass'],
    drop_first=True,
    dtype=int
)

In [41]:
[c for c in housing.columns if c.startswith('MS.SubClass_')]

['MS.SubClass_30',
 'MS.SubClass_40',
 'MS.SubClass_45',
 'MS.SubClass_50',
 'MS.SubClass_60',
 'MS.SubClass_70',
 'MS.SubClass_75',
 'MS.SubClass_80',
 'MS.SubClass_85',
 'MS.SubClass_90',
 'MS.SubClass_120',
 'MS.SubClass_150',
 'MS.SubClass_160',
 'MS.SubClass_180',
 'MS.SubClass_190']

In [42]:
housing = pd.get_dummies(
    housing,
    columns=['MS.Zoning'],
    drop_first=True,
    dtype=int
)

In [43]:
[c for c in housing.columns if c.startswith('MS.Zoning_')]

['MS.Zoning_C (all)',
 'MS.Zoning_FV',
 'MS.Zoning_I (all)',
 'MS.Zoning_RH',
 'MS.Zoning_RL',
 'MS.Zoning_RM']

In [44]:
housing = pd.get_dummies(
    housing,
    columns=['Lot.Shape'],
    drop_first=True,
    dtype=int
)

In [45]:
[c for c in housing.columns if c.startswith('Lot.Shape_')]

['Lot.Shape_IR2', 'Lot.Shape_IR3', 'Lot.Shape_Reg']

In [46]:
housing = pd.get_dummies(
    housing,
    columns=['Land.Contour'],
    drop_first=True,
    dtype=int
)

In [47]:
[c for c in housing.columns if c.startswith('Land.Contour_')]

['Land.Contour_HLS', 'Land.Contour_Low', 'Land.Contour_Lvl']

In [48]:
housing = pd.get_dummies(
    housing,
    columns=['Lot.Config'],
    drop_first=True,
    dtype=int
)

In [49]:
[c for c in housing.columns if c.startswith('Lot.Config_')]

['Lot.Config_CulDSac', 'Lot.Config_FR2', 'Lot.Config_FR3', 'Lot.Config_Inside']

In [50]:
housing = pd.get_dummies(
    housing,
    columns=['Foundation'],
    drop_first=True,
    dtype=int
)

In [51]:
[c for c in housing.columns if c.startswith('Foundation_')]

['Foundation_CBlock',
 'Foundation_PConc',
 'Foundation_Slab',
 'Foundation_Stone',
 'Foundation_Wood']

In [52]:
housing = pd.get_dummies(
    housing,
    columns=['Electrical'],
    drop_first=True,
    dtype=int
)

In [53]:
[c for c in housing.columns if c.startswith('Electrical_')]

['Electrical_FuseF', 'Electrical_FuseP', 'Electrical_Mix', 'Electrical_SBrkr']

In [54]:
housing = pd.get_dummies(
    housing,
    columns=['Garage.Type'],
    drop_first=True,
    dtype=int
)

In [55]:
[c for c in housing.columns if c.startswith('Garage.Type_')]

['Garage.Type_Attchd',
 'Garage.Type_Basment',
 'Garage.Type_BuiltIn',
 'Garage.Type_CarPort',
 'Garage.Type_Detchd']

In [56]:
housing = pd.get_dummies(
    housing,
    columns=['Sale.Condition'],
    drop_first=True,
    dtype=int
)

In [57]:
[c for c in housing.columns if c.startswith('Sale.Condition_')]

['Sale.Condition_AdjLand',
 'Sale.Condition_Alloca',
 'Sale.Condition_Family',
 'Sale.Condition_Normal',
 'Sale.Condition_Partial']

In [58]:
housing = pd.get_dummies(
    housing,
    columns=['Exterior.1st'],
    drop_first=True,
    dtype=int
)

In [59]:
[c for c in housing.columns if c.startswith('Exterior.1st_')]

['Exterior.1st_AsphShn',
 'Exterior.1st_BrkComm',
 'Exterior.1st_BrkFace',
 'Exterior.1st_CBlock',
 'Exterior.1st_CemntBd',
 'Exterior.1st_HdBoard',
 'Exterior.1st_ImStucc',
 'Exterior.1st_MetalSd',
 'Exterior.1st_Plywood',
 'Exterior.1st_PreCast',
 'Exterior.1st_Stone',
 'Exterior.1st_Stucco',
 'Exterior.1st_VinylSd',
 'Exterior.1st_Wd Sdng',
 'Exterior.1st_WdShing']

In [60]:
housing = pd.get_dummies(
    housing,
    columns=['Condition.1'],
    drop_first=True,
    dtype=int
)

In [61]:
exter_qual_map = {
    'Po': 1,
    'Fa': 2, 
    'TA': 3,
    'Gd': 4,
    'Ex': 5
}

housing['Exter.Qual'] = housing['Exter.Qual'].map(exter_qual_map)

housing['Exter.Qual'].value_counts().sort_index()

Exter.Qual
2      35
3    1799
4     989
5     107
Name: count, dtype: int64

## 12. Standardize Predictor Variables

The predictor variables are measured on very different scales and ranges. I think standardization will be the best choice to put the predictors on a common scale. Espcially since I am thinking about using Lasso Regression for this model.

In [62]:
from sklearn.preprocessing import StandardScaler

X = housing.drop(columns=['SalePrice'])
y = housing['SalePrice']

scaler = StandardScaler()

X_standardized = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

In [65]:
housing_standardized = pd.concat(
    [X_standardized, y],
    axis=1
)

In [66]:
X_standardized.describe().loc[['mean', 'std']]

,Lot.Frontage,Lot.Area,Overall.Qual,Overall.Cond,Mas.Vnr.Area,Exter.Qual,Central.Air,Functional,Gr.Liv.Area,Bedroom.AbvGr,...,Exterior.1st_Wd Sdng,Exterior.1st_WdShing,Condition.1_Feedr,Condition.1_Norm,Condition.1_PosA,Condition.1_PosN,Condition.1_RRAe,Condition.1_RRAn,Condition.1_RRNe,Condition.1_RRNn
mean,2.825196e-16,5.820145e-17,-8.002700e-17,2.691817e-16,3.880097e-17,2.182554e-17,-4.243856e-17,3.152579e-17,3.637591e-17,1.697542e-17,...,-2.182554e-17,-4.850121e-18,2.425061e-18,2.910073e-17,4.850121e-18,3.395085e-17,4.001350e-17,2.182554e-17,-1.455036e-17,1.576289e-17
std,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,...,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00,1.000171e+00


# Final Comments

All features needed for the model have been ensured for model readiness. Some of have been converted into numerical values, some new features have been created, all features have been standardized. The dataset is model ready. 

In [67]:
housing_standardized.to_csv(
    'ames_housing_feature_engineered.csv',
    index=False
)